<a href="https://colab.research.google.com/github/sairas2124/Gradient_descent-/blob/main/datacleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# KAGGLE DATA CLEANING COURSE: ALL-IN-ONE MASTER PIPELINE
# Dataset: Titanic (Simulated with advanced real-world corruptions)
# ==============================================================================

import pandas as pd
import numpy as np
from scipy import stats
import charset_normalizer  # Used for Lesson 4: Character Encodings

# --- SETUP: Load Data & Intentionally Add Corruptions for Learning ---
# (We load standard data and inject raw errors to demonstrate all 5 Kaggle tasks)
np.random.seed(42)
raw_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(raw_url)

# Injecting artificial date column, encoding bugs, and typos for the tutorial
df['Signup_Date'] = pd.date_range(start='1/1/2024', periods=len(df), freq='h').strftime('%d/%m/%Y')
df.loc[np.random.choice(df.index, 50), 'Signup_Date'] = '2024-05-12' # Mix formats
df['Embark_Port'] = df['Embarked'].map({'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'})
df.loc[np.random.choice(df.index, 20), 'Embark_Port'] = 'southampton ' # Add typos/spaces


# ==============================================================================
# LESSON 1: HANDLING MISSING VALUES
# ==============================================================================
print("--- LESSON 1: HANDLING MISSING VALUES ---")

# Step A: Identify missing points
missing_counts = df.isnull().sum()
print(f"Missing values per column:\n{missing_counts[missing_counts > 0]}\n")

# Step B: Strategic Imputation
# 1. Numerical columns (Age): Drop rows if missingness is completely random, or fill with median
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)

# 2. Categorical columns (Cabin): Too many missing values, fill with a new structural category
df['Cabin'] = df['Cabin'].fillna('Unknown')

# 3. Drop rows with critical target missingness (Embarked has only 2 missing rows)
df = df.dropna(subset=['Embarked'])
print(f"Remaining missing values in Age/Cabin/Embarked: {df[['Age', 'Cabin', 'Embarked']].isnull().sum().sum()}\n")


# ==============================================================================
# LESSON 2: SCALING AND NORMALIZATION
# ==============================================================================
print("--- LESSON 2: SCALING AND NORMALIZATION ---")

# 1. Scaling: Transform range to [0, 1] (Great for models like KNN, SVM, Neural Nets)
# Formula: (x - min) / (max - min)
fare_min = df['Fare'].min()
fare_max = df['Fare'].max()
df['Fare_Scaled'] = (df['Fare'] - fare_min) / (fare_max - fare_min)

# 2. Normalization: Change the distribution shape to look like a Gaussian Normal Curve (Bell Curve)
# Essential for linear regression or algorithms assuming normal distributions. We use Box-Cox (requires data > 0)
# Add a tiny constant to prevent 0 values in Fare
positive_fare = df['Fare'] + 0.001
df['Fare_Normalized'] = stats.boxcox(positive_fare)[0]

print(f"Original Fare Max: {df['Fare'].max()} -> Scaled Max: {df['Fare_Scaled'].max()}")
print(f"Original skew: {df['Fare'].skew():.2f} -> Normalized skew: {df['Fare_Normalized'].skew():.2f}\n")


# ==============================================================================
# LESSON 3: PARSING DATES
# ==============================================================================
print("--- LESSON 3: PARSING DATES ---")

# Python objects (strings) must be converted into datetime64 format for time-series extraction
# Using format='mixed' handles datasets with mismatched, messy dates seamlessly
df['Signup_Date_Cleaned'] = pd.to_datetime(df['Signup_Date'], format='mixed')

# Now we can extract structural time elements easily
df['Signup_Day'] = df['Signup_Date_Cleaned'].dt.day
df['Signup_Month'] = df['Signup_Date_Cleaned'].dt.month

print("Sample parsed dates verification:")
print(df[['Signup_Date', 'Signup_Date_Cleaned', 'Signup_Day']].head(3), "\n")


# ==============================================================================
# LESSON 4: CHARACTER ENCODINGS
# ==============================================================================
print("--- LESSON 4: CHARACTER ENCODINGS ---")

# Sometimes files crash with UnicodeDecodeError due to specific formats (like Windows-1252 or ISO-8859-1)
# Step A: Simulate a broken file save using specific encoding
mock_string = "Anish,Cherbourg-TGV,München"
bytes_data = mock_string.encode('utf-8')

# Step B: Automatically detect unknown encoding bytes safely
detection = charset_normalizer.detect(bytes_data)
detected_encoding = detection['encoding']
print(f"Detected safe data encoding format: {detected_encoding}")

# Step C: Decode smoothly back to ordinary Python string text
clean_string = bytes_data.decode(detected_encoding)
print(f"Decoded Clean Output: {clean_string}\n")


# ==============================================================================
# LESSON 5: INCONSISTENT DATA ENTRY
# ==============================================================================
print("--- LESSON 5: INCONSISTENT DATA ENTRY ---")

# Examine unique values with lowercase/whitespace messiness
print("Before cleaning Embark_Port:", df['Embark_Port'].unique()[:4])

# Step A: Standardize casing and strip leading/trailing whitespace buffers
df['Embark_Port'] = df['Embark_Port'].str.lower().str.strip()

# Step B: Map or replace lingering near-matches
# (e.g., if there were typos like 'southamp' vs 'southampton')
port_mapping = {'southampton': 'Southampton', 'cherbourg': 'Cherbourg', 'queenstown': 'Queenstown'}
df['Embark_Port'] = df['Embark_Port'].map(port_mapping)

print("After cleaning Embark_Port:", df['Embark_Port'].unique()[:3])
print("\n==============================================================================")
print("ALL 5 KAGGLE DATA CLEANING STEPS COMPLETED IN A SINGLE WORKFLOW SUCCESSFULLY!")
print("==============================================================================")

--- LESSON 1: HANDLING MISSING VALUES ---
Missing values per column:
Age            177
Cabin          687
Embarked         2
Embark_Port      2
dtype: int64

Remaining missing values in Age/Cabin/Embarked: 0

--- LESSON 2: SCALING AND NORMALIZATION ---
Original Fare Max: 512.3292 -> Scaled Max: 1.0
Original skew: 4.80 -> Normalized skew: 0.33

--- LESSON 3: PARSING DATES ---
Sample parsed dates verification:
  Signup_Date Signup_Date_Cleaned  Signup_Day
0  01/01/2024          2024-01-01           1
1  01/01/2024          2024-01-01           1
2  01/01/2024          2024-01-01           1 

--- LESSON 4: CHARACTER ENCODINGS ---
Detected safe data encoding format: utf-8
Decoded Clean Output: Anish,Cherbourg-TGV,München

--- LESSON 5: INCONSISTENT DATA ENTRY ---
Before cleaning Embark_Port: ['Southampton' 'Cherbourg' 'Queenstown' 'southampton ']
After cleaning Embark_Port: ['Southampton' 'Cherbourg' 'Queenstown']

ALL 5 KAGGLE DATA CLEANING STEPS COMPLETED IN A SINGLE WORKFLOW SUCCESSFU